> **Datos (acuerdo del grupo, octubre de 2026):** cargar con `from src.data import make_dataset, split_df, load_class_weights` en lugar de `tc.make_dataset`, `tc.compute_class_weights` y `tc.load_split` (ver `docs/02_flujo_de_trabajo.md`, apartado 9). Pendiente de adaptar por el Módulo 2.

# Módulo 2 — Fine-tuning y evaluación cuantitativa comparativa
**TFM · Clasificación de lesiones cutáneas (HAM10000) con CNN y Grad-CAM**
Responsable: **Alex Jessiel Reinoso González** · Objetivos específicos: **OE2, OE3, OE4**

### Protocolo experimental
- Mismo split (`split_v1.csv`, Módulo 1), mismo preprocesado y mismos hiperparámetros base para las 3 arquitecturas.
- Fine-tuning en fases: (1) backbone congelado → entrenar cabeza; (2) descongelar último bloque, LR bajo; (3) descongelar penúltimo bloque, LR más bajo.
- **3 semillas por modelo** → se reporta media ± desviación típica.
- **Regla de selección (en validación):** sensibilidad de melanoma ≥ 80 % como requisito mínimo y, entre los que la cumplen, mayor **macro-F1**. Se aplica a los checkpoints por época y a la elección del mejor modelo.
- **Test se usa una única vez por modelo**, al final y con la configuración ya cerrada.
- Métricas en test: accuracy, balanced accuracy, macro-F1, AUC-ROC macro one-vs-rest, precisión/sensibilidad/especificidad por clase (one-vs-rest), matriz de confusión.
- Todo run queda registrado en **MLflow (DagsHub)**; modelos y predicciones también en Drive.

### Contrato de salida (para el Módulo 3)
`TFM/models/{arch}_seed{s}_best.keras` · `TFM/results/{arch}_seed{s}_test_preds.csv` · `TFM/results/representative_models.json` · `TFM/results/mel_false_negatives.csv`

In [ ]:
# @title 0. Entorno: Drive, repositorio y dependencias
# Requisitos (una sola vez por persona):
#  - En Drive: "Añadir acceso directo" de la carpeta compartida TFM a "Mi unidad".
#  - En Colab > Secretos (icono de llave): DAGSHUB_TOKEN y, si el repo es privado, GITHUB_TOKEN.
#  - Entorno de ejecución > Cambiar tipo > GPU T4.
import os, sys, subprocess
from google.colab import drive, userdata

drive.mount("/content/drive")

GITHUB_USER = "aReinoso007"
REPO_NAME = "tfm-melanoma-cnn-gradcam"
REPO_DIR = f"/content/{REPO_NAME}"
DRIVE_ROOT = "/content/drive/MyDrive/TFM"   # carpeta compartida del grupo

try:
    _tok = userdata.get("GITHUB_TOKEN")
except Exception:
    _tok = None
_url = f"https://{_tok}@github.com/{GITHUB_USER}/{REPO_NAME}.git" if _tok else \
       f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", _url, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
sys.path.insert(0, f"{REPO_DIR}/src")

!pip -q install -r {REPO_DIR}/requirements.txt kagglehub   # MLflow 3.x, como requirements.txt

import importlib, tfm_common as tc
importlib.reload(tc)
paths = tc.get_paths(DRIVE_ROOT)
print("Repo:", REPO_DIR, "| Drive:", paths.drive_root)

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Configuración del experimento

In [ ]:
# @title 1.1 Hiperparámetros (idénticos para las 3 arquitecturas)
CONFIG = dict(
    img_size=tc.IMG_SIZE,
    batch_size=32,
    dropout=0.3,
    label_smoothing=0.0,
    mixed_precision=True,            # acelera en T4; la cabeza se mantiene en float32
    class_weight_mode="balanced",    # "balanced" | "sqrt" | None
    aug_mode="selective",            # "selective" | "all" | "none"
    aug_classes=[c for c in tc.CLASSES if c != "nv"],
    oversample=None,                 # p. ej. {"df": 4, "vasc": 4} (recalcula pesos)
    min_mel_sens=tc.MIN_MEL_SENSITIVITY,
    phases=[
        dict(name="f1_cabeza",          unfreeze_stage=None, lr=1e-3, epochs=10, patience=3),
        dict(name="f2_ultimo_bloque",   unfreeze_stage=0,    lr=3e-5, epochs=15, patience=4),
        dict(name="f3_penultimo_bloque", unfreeze_stage=1,   lr=1e-5, epochs=10, patience=4),
    ],
    lr_reduce_factor=0.5, lr_reduce_patience=2,
)
CONFIG

## 2. Datos

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, time, shutil, json
import tensorflow as tf
keras = tf.keras
sns.set_theme(style="whitegrid")
FIG_DIR = paths.drive_figures / "modulo2"; FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Imágenes locales:", tc.ensure_local_images(paths))
df = tc.load_split(paths)
train_df_base = df[df.split == "train"].reset_index(drop=True)
val_df = df[df.split == "val"].reset_index(drop=True)
test_df = df[df.split == "test"].reset_index(drop=True)
train_df = tc.oversample(train_df_base, CONFIG["oversample"])
class_weight = tc.compute_class_weights(train_df, CONFIG["class_weight_mode"])
print({k: len(v) for k, v in [("train", train_df), ("val", val_df), ("test", test_df)]})
print("Pesos de clase:", {tc.CLASSES[k]: round(v, 2) for k, v in (class_weight or {}).items()})

## 3. Callback de validación, checkpoint y early stopping

In [ ]:
class ValSelectionCallback(keras.callbacks.Callback):
    """En cada época: predice validación, calcula macro-F1 / sensibilidad de melanoma,
    guarda el mejor checkpoint según la regla de selección, aplica early stopping y
    reducción de LR, y registra en MLflow. Su estado se comparte entre fases."""

    def __init__(self, val_ds, y_val, ckpt_path, state, phase, patience, min_mel_sens,
                 lr_factor=0.5, lr_patience=2):
        super().__init__()
        self.val_ds, self.y_val, self.ckpt_path, self.state = val_ds, y_val, ckpt_path, state
        self.phase, self.patience, self.min_mel_sens = phase, patience, min_mel_sens
        self.lr_factor, self.lr_patience = lr_factor, lr_patience
        self.phase_best, self.wait, self.lr_wait = None, 0, 0

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        probs = self.model.predict(self.val_ds, verbose=0)
        m, _, _ = tc.compute_metrics(self.y_val, probs)
        p = np.clip(probs[np.arange(len(self.y_val)), self.y_val], 1e-7, 1)
        val_loss = float(-np.log(p).mean())
        score = tc.selection_score(m, self.min_mel_sens)
        self.state["global_epoch"] += 1
        ge = self.state["global_epoch"]
        lr = float(keras.ops.convert_to_numpy(self.model.optimizer.learning_rate))
        rec = dict(epoch=ge, phase=self.phase, lr=lr, loss=float(logs.get("loss", np.nan)),
                   accuracy=float(logs.get("accuracy", np.nan)), val_loss=val_loss,
                   val_accuracy=m["accuracy"], val_macro_f1=m["macro_f1"],
                   val_bal_acc=m["balanced_accuracy"], val_mel_sens=m["mel_sensitivity"],
                   val_auc=m["auc_macro_ovr"])
        self.state["history"].append(rec)
        tc.safe(mlflow.log_metrics, {k: v for k, v in rec.items() if k not in ("epoch", "phase")}, step=ge)

        improved_global = self.state["best_score"] is None or score > tuple(self.state["best_score"])
        if improved_global:
            self.state.update(best_score=list(score), best_epoch=ge, best_phase=self.phase)
            self.model.save(self.ckpt_path)
        # early stopping / reduce LR dentro de la fase
        if self.phase_best is None or score > self.phase_best:
            self.phase_best, self.wait, self.lr_wait = score, 0, 0
        else:
            self.wait += 1; self.lr_wait += 1
            if self.lr_wait >= self.lr_patience:
                self.model.optimizer.learning_rate.assign(lr * self.lr_factor); self.lr_wait = 0
            if self.wait >= self.patience:
                self.model.stop_training = True
        flag = " *" if improved_global else ""
        print(f"[{self.phase}] ep {ge:02d} loss={rec['loss']:.3f} val_loss={val_loss:.3f} "
              f"macroF1={m['macro_f1']:.3f} melSens={m['mel_sensitivity']:.3f} balAcc={m['balanced_accuracy']:.3f}{flag}")

## 4. Función de experimento (una arquitectura × una semilla)

In [ ]:
def run_name_of(arch, seed, suffix=""):
    return f"{arch}_seed{seed}{suffix}"


def run_experiment(arch, seed, cfg=CONFIG, quick=False, force=False):
    """Entrena en fases, guarda el mejor checkpoint (regla de selección) y evalúa en validación.
    Reanudable: si ya hay resultados en Drive, no repite."""
    suffix = "_smoke" if quick else ""
    name = run_name_of(arch, seed, suffix)
    model_path = paths.drive_models / f"{name}_best.keras"
    val_json = paths.drive_results / f"{name}_val.json"
    if val_json.exists() and model_path.exists() and not force:
        print(f"[skip] {name} ya entrenado"); return tc.load_json(val_json)

    tc.set_seed(seed)
    tr, va = (train_df.sample(400, random_state=seed), val_df.sample(300, random_state=seed)) if quick else (train_df, val_df)
    train_ds = tc.make_dataset(tr, arch, training=True, batch_size=cfg["batch_size"], seed=seed,
                               aug_mode=cfg["aug_mode"], aug_classes=cfg["aug_classes"])
    val_ds = tc.make_dataset(va, arch, training=False, batch_size=cfg["batch_size"])
    y_val = va.label.values

    model = tc.build_model(arch, dropout=cfg["dropout"], mixed_precision=cfg["mixed_precision"])
    local_ckpt = str(paths.local_ckpt / f"{name}.keras")
    state = dict(best_score=None, best_epoch=None, best_phase=None, global_epoch=0, history=[])
    t0 = time.time()

    with mlflow.start_run(run_name=name) as run:
        mlflow.set_tags({"modulo": "2", "arch": arch, "seed": seed, "smoke": quick,
                         "split_version": tc.SPLIT_VERSION})
        tc.safe(mlflow.log_params, {k: v for k, v in cfg.items() if k != "phases"} | {"arch": arch, "seed": seed})
        tc.safe(mlflow.log_dict, cfg, "config.json")

        for ph in cfg["phases"]:
            stage = None if ph["unfreeze_stage"] is None else tc.UNFREEZE_STAGES[arch][ph["unfreeze_stage"]]
            info = tc.set_trainable(model, stage)
            print(f"\n== {name} · {ph['name']} · desde {stage} · entrenables {info['trainable_params']:,}/{info['total_params']:,}")
            model.compile(optimizer=keras.optimizers.Adam(ph["lr"]),
                          loss=keras.losses.CategoricalCrossentropy(label_smoothing=cfg["label_smoothing"]),
                          metrics=["accuracy"])
            cb = ValSelectionCallback(val_ds, y_val, local_ckpt, state, ph["name"], ph["patience"],
                                      cfg["min_mel_sens"], cfg["lr_reduce_factor"], cfg["lr_reduce_patience"])
            model.fit(train_ds, epochs=1 if quick else ph["epochs"], class_weight=class_weight,
                      callbacks=[cb], verbose=0)
            # la siguiente fase parte del mejor punto alcanzado hasta ahora
            model.set_weights(keras.models.load_model(local_ckpt).get_weights())
            shutil.copy(local_ckpt, model_path)

        best = keras.models.load_model(local_ckpt)
        probs = best.predict(val_ds, verbose=0)
        m, per_class, cm = tc.compute_metrics(y_val, probs)
        train_time = time.time() - t0

        out = dict(arch=arch, seed=seed, run_id=run.info.run_id, best_epoch=state["best_epoch"],
                   best_phase=state["best_phase"], train_time_s=train_time,
                   n_params=int(best.count_params()), val=m, history=state["history"])
        tc.save_json(out, val_json)
        pd.DataFrame(probs, columns=[f"p_{c}" for c in tc.CLASSES]).assign(
            image_id=va.image_id.values, y_true=y_val, y_pred=probs.argmax(1)
        ).to_csv(paths.drive_results / f"{name}_val_preds.csv", index=False)

        tc.safe(mlflow.log_metrics, {f"final_val_{k}": v for k, v in m.items()})
        tc.safe(mlflow.log_metrics, {"train_time_s": train_time, "best_epoch": state["best_epoch"]})
        fig = tc.plot_confusion_matrix(cm, title=f"{name} · validación"); tc.safe(mlflow.log_figure, fig, "cm_val.png"); plt.close(fig)
        tc.safe(mlflow.log_artifact, str(val_json))
        # Modelo en MLflow (puede tardar; Drive es la copia de referencia)
        if not quick:
            tc.safe(mlflow.log_artifact, str(model_path), "model")
    print(f"{name}: macroF1={m['macro_f1']:.3f} melSens={m['mel_sensitivity']:.3f} ({train_time/60:.1f} min)")
    return out

## 5. Prueba rápida (smoke test)
Ejecutar antes de lanzar el entrenamiento completo: 1 época por fase con subconjuntos pequeños. No toca resultados oficiales.

In [ ]:
mlflow = tc.init_mlflow(tc.MLFLOW_EXPERIMENTS["sandbox"])
for arch in tc.ARCHS:
    run_experiment(arch, tc.SEEDS[0], quick=True, force=True)

## 6. Entrenamiento completo: 3 arquitecturas × 3 semillas
Reanudable: si Colab se desconecta, volver a ejecutar desde el inicio; los runs completados se saltan.
Se puede repartir entre cuentas ejecutando subconjuntos de `ARCHS`/`SEEDS`.

In [ ]:
mlflow = tc.init_mlflow(tc.MLFLOW_EXPERIMENT)
ARCHS_TO_RUN = tc.ARCHS          # p. ej. ["resnet50"]
SEEDS_TO_RUN = tc.SEEDS
val_results = {}
for arch in ARCHS_TO_RUN:
    for seed in SEEDS_TO_RUN:
        val_results[(arch, seed)] = run_experiment(arch, seed)

## 7. Curvas de aprendizaje y comparación en validación

In [ ]:
def load_val(arch, seed):
    p = paths.drive_results / f"{run_name_of(arch, seed)}_val.json"
    return tc.load_json(p) if p.exists() else None

fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=False)
for j, arch in enumerate(tc.ARCHS):
    for seed in tc.SEEDS:
        r = load_val(arch, seed)
        if not r: continue
        h = pd.DataFrame(r["history"])
        axes[0, j].plot(h.epoch, h.loss, label=f"train s{seed}"); axes[0, j].plot(h.epoch, h.val_loss, "--", label=f"val s{seed}")
        axes[1, j].plot(h.epoch, h.val_macro_f1, label=f"macro-F1 s{seed}"); axes[1, j].plot(h.epoch, h.val_mel_sens, ":", label=f"sens. mel s{seed}")
        for e in h.groupby("phase").epoch.min().values[1:]:
            axes[0, j].axvline(e - 0.5, color="grey", lw=0.6); axes[1, j].axvline(e - 0.5, color="grey", lw=0.6)
    axes[0, j].set_title(f"{arch} · pérdida"); axes[1, j].set_title(f"{arch} · validación")
    axes[1, j].axhline(tc.MIN_MEL_SENSITIVITY, color="red", lw=0.8)
    axes[0, j].legend(fontsize=7); axes[1, j].legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG_DIR / "curvas_aprendizaje.png", dpi=200); plt.show()

In [ ]:
# @title 7.2 Resumen en validación (media ± std) y modelo representativo por arquitectura
KEYS = ["accuracy", "balanced_accuracy", "macro_f1", "auc_macro_ovr", "mel_sensitivity", "mel_specificity"]
rows = []
for arch in tc.ARCHS:
    for seed in tc.SEEDS:
        r = load_val(arch, seed)
        if r: rows.append(dict(arch=arch, seed=seed, best_epoch=r["best_epoch"], best_phase=r["best_phase"],
                               train_min=r["train_time_s"] / 60, n_params=r["n_params"], **{k: r["val"][k] for k in KEYS}))
val_tab = pd.DataFrame(rows)
display(val_tab.round(4))

def mean_std(tab, keys):
    g = tab.groupby("arch")[keys]
    return (g.mean().round(4).astype(str) + " ± " + g.std().round(4).astype(str)).reindex(tc.ARCHS)
display(mean_std(val_tab, KEYS + ["train_min"]))

# Modelo representativo por arquitectura (para el Módulo 3): mejor semilla según la regla
representative = {}
for arch, g in val_tab.groupby("arch"):
    best = max(g.itertuples(), key=lambda r: tc.selection_score(r._asdict()))
    representative[arch] = dict(seed=int(best.seed), model=str(paths.drive_models / f"{run_name_of(arch, best.seed)}_best.keras"),
                                test_preds=str(paths.drive_results / f"{run_name_of(arch, best.seed)}_test_preds.csv"))
tc.save_json(representative, paths.representative_json)
representative

## 8. Evaluación en test (una sola vez por modelo)
> Ejecutar **solo** cuando la configuración esté cerrada. Cada evaluación deja una marca (`*_test.json`); re-evaluar exige `force=True` y debe justificarse en la memoria.

In [ ]:
def evaluate_test(arch, seed, force=False):
    name = run_name_of(arch, seed)
    test_json = paths.drive_results / f"{name}_test.json"
    if test_json.exists() and not force:
        return tc.load_json(test_json)
    val_info = load_val(arch, seed)
    model = keras.models.load_model(paths.drive_models / f"{name}_best.keras")
    test_ds = tc.make_dataset(test_df, arch, training=False, batch_size=CONFIG["batch_size"])
    probs = model.predict(test_ds, verbose=0)
    m, per_class, cm = tc.compute_metrics(test_df.label.values, probs)
    preds = pd.DataFrame(probs, columns=[f"p_{c}" for c in tc.CLASSES])
    preds.insert(0, "image_id", test_df.image_id.values)
    preds.insert(1, "lesion_id", test_df.lesion_id.values)
    preds.insert(2, "y_true", test_df.label.values)
    preds.insert(3, "y_pred", probs.argmax(1))
    preds.insert(4, "dx_true", test_df.dx.values)
    preds.insert(5, "dx_pred", [tc.CLASSES[i] for i in probs.argmax(1)])
    preds.to_csv(paths.drive_results / f"{name}_test_preds.csv", index=False)
    out = dict(arch=arch, seed=seed, test=m, per_class=per_class.reset_index().to_dict("records"), cm=cm.tolist())
    tc.save_json(out, test_json)
    with mlflow.start_run(run_id=val_info["run_id"]):
        tc.safe(mlflow.log_metrics, {f"test_{k}": v for k, v in m.items()})
        fig = tc.plot_confusion_matrix(cm, title=f"{name} · test"); tc.safe(mlflow.log_figure, fig, "cm_test.png"); plt.close(fig)
        tc.safe(mlflow.log_artifact, str(paths.drive_results / f"{name}_test_preds.csv"))
    return out

test_results = {(a, s): evaluate_test(a, s) for a in tc.ARCHS for s in tc.SEEDS
                if (paths.drive_models / f"{run_name_of(a, s)}_best.keras").exists()}

In [ ]:
# @title 8.2 Tabla comparativa final en test (media ± std sobre 3 semillas)
test_tab = pd.DataFrame([dict(arch=a, seed=s, **r["test"]) for (a, s), r in test_results.items()])
TEST_KEYS = ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "auc_macro_ovr",
             "macro_precision", "macro_specificity", "mel_sensitivity", "mel_specificity", "mel_precision"]
summary_test = mean_std(test_tab, TEST_KEYS)
display(summary_test)
summary_test.to_csv(paths.drive_results / "resumen_test_media_std.csv")

# Por clase: sensibilidad / especificidad / precisión / F1 (media sobre semillas)
pc = pd.concat([pd.DataFrame(r["per_class"]).assign(arch=a, seed=s) for (a, s), r in test_results.items()])
pc_mean = pc.groupby(["arch", "clase"])[["precision", "sensibilidad", "especificidad", "f1", "auc"]].mean().round(3)
display(pc_mean)
pc_mean.to_csv(paths.drive_results / "por_clase_test_media.csv")

fig, ax = plt.subplots(figsize=(12, 4))
sns.barplot(data=pc, x="clase", y="sensibilidad", hue="arch", order=tc.CLASSES, ax=ax, errorbar="sd")
ax.axhline(tc.MIN_MEL_SENSITIVITY, color="red", lw=0.8); ax.set_title("Sensibilidad por clase en test (media ± sd, 3 semillas)")
fig.tight_layout(); fig.savefig(FIG_DIR / "sensibilidad_por_clase_test.png", dpi=200); plt.show()

In [ ]:
# @title 8.3 Matrices de confusión y curvas ROC (modelo representativo de cada arquitectura)
from sklearn.metrics import roc_curve, auc
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for ax, arch in zip(axes, tc.ARCHS):
    s = representative[arch]["seed"]; cm = np.array(test_results[(arch, s)]["cm"])
    cmn = cm / cm.sum(1, keepdims=True)
    sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues", cbar=False, xticklabels=tc.CLASSES, yticklabels=tc.CLASSES, ax=ax)
    ax.set_title(f"{arch} (seed {s})"); ax.set_xlabel("Predicción"); ax.set_ylabel("Real")
fig.tight_layout(); fig.savefig(FIG_DIR / "matrices_confusion_test.png", dpi=200); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, arch in zip(axes, tc.ARCHS):
    p = pd.read_csv(representative[arch]["test_preds"])
    for k, c in enumerate(tc.CLASSES):
        fpr, tpr, _ = roc_curve(p.y_true == k, p[f"p_{c}"])
        ax.plot(fpr, tpr, lw=2.5 if c == "mel" else 1, label=f"{c} (AUC={auc(fpr, tpr):.2f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.6); ax.set_title(f"ROC one-vs-rest · {arch}"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "roc_test.png", dpi=200); plt.show()

## 9. OE4 — Análisis de falsos negativos de melanoma
Falso negativo (FN) = lesión con `dx = mel` clasificada como cualquier otra clase. Se analiza: cuántos hay por modelo, a qué clase se confunden, con qué confianza, cuáles fallan en **todos** los modelos (casos difíciles) y si difieren de los aciertos en metadatos.

In [ ]:
meta_cols = ["image_id", "age", "sex", "localization", "dx_type"]
fn_rows = []
for (a, s) in test_results:
    p = pd.read_csv(paths.drive_results / f"{run_name_of(a, s)}_test_preds.csv")
    mel = p[p.y_true == tc.MEL_IDX].assign(arch=a, seed=s)
    mel["fn"] = mel.y_pred != tc.MEL_IDX
    fn_rows.append(mel)
mel_all = pd.concat(fn_rows).merge(df[meta_cols], on="image_id", how="left")

print("FN de melanoma por modelo:")
display(mel_all.groupby(["arch", "seed"]).fn.agg(["sum", "count"]).assign(tasa=lambda d: (d["sum"] / d["count"]).round(3)))

print("¿A qué clase se confunden? (todos los modelos)")
display(pd.crosstab(mel_all[mel_all.fn].arch, mel_all[mel_all.fn].dx_pred, normalize="index").round(3))

n_models = mel_all.groupby(["arch", "seed"]).ngroups
hard = mel_all.groupby("image_id").fn.sum().rename("n_modelos_fn").reset_index()
hard["fn_en_todos"] = hard.n_modelos_fn == n_models
print(f"Melanomas en test: {len(hard)} | FN en todos los modelos: {hard.fn_en_todos.sum()} | FN en ninguno: {(hard.n_modelos_fn == 0).sum()}")
fig, ax = plt.subplots(figsize=(7, 3.5)); hard.n_modelos_fn.value_counts().sort_index().plot.bar(ax=ax)
ax.set_xlabel("Nº de modelos que fallan la imagen"); ax.set_ylabel("Nº de melanomas"); fig.tight_layout()
fig.savefig(FIG_DIR / "fn_melanoma_consistencia.png", dpi=200); plt.show()

In [ ]:
# @title 9.2 Confianza y metadatos: FN vs aciertos (TP)
mel_all["p_mel_asignada"] = mel_all["p_mel"]
display(mel_all.groupby(["arch", "fn"]).p_mel_asignada.describe()[["mean", "50%", "min", "max"]].round(3))

uni = mel_all.drop_duplicates("image_id").merge(hard, on="image_id")
uni["grupo"] = np.where(uni.fn_en_todos, "FN en todos", np.where(uni.n_modelos_fn == 0, "acierto en todos", "mixto"))
display(uni.groupby("grupo").age.describe().round(1))
display(pd.crosstab(uni.grupo, uni.dx_type, normalize="index").round(2))
display(pd.crosstab(uni.localization, uni.grupo))

# Exportar para el Módulo 3 (Grad-CAM de los fallos críticos)
export = mel_all[mel_all.fn][["image_id", "arch", "seed", "dx_pred", "p_mel", "age", "sex", "localization", "dx_type"]] \
    .merge(hard, on="image_id")
export.to_csv(paths.drive_results / "mel_false_negatives.csv", index=False)
print("Exportado:", paths.drive_results / "mel_false_negatives.csv", len(export), "filas")

## 10. (Opcional) Intervalos de confianza bootstrap y test de McNemar
Si el tiempo lo permite. Bootstrap por **lesión** (respeta la dependencia entre imágenes de la misma lesión) sobre el modelo representativo; McNemar pareado entre arquitecturas sobre el mismo test.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import f1_score

def bootstrap_ci(p, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed)
    lesions = p.lesion_id.unique()
    idx_by_lesion = p.groupby("lesion_id").indices
    stats = []
    for _ in range(n_boot):
        sel = np.concatenate([idx_by_lesion[l] for l in rng.choice(lesions, len(lesions), replace=True)])
        q = p.iloc[sel]
        mel = q.y_true == tc.MEL_IDX
        stats.append((f1_score(q.y_true, q.y_pred, average="macro", labels=range(tc.NUM_CLASSES), zero_division=0),
                      (q.y_pred[mel] == tc.MEL_IDX).mean()))
    s = np.array(stats)
    return {"macro_f1_IC95": np.percentile(s[:, 0], [2.5, 97.5]).round(3).tolist(),
            "mel_sens_IC95": np.percentile(s[:, 1], [2.5, 97.5]).round(3).tolist()}

preds_rep = {a: pd.read_csv(representative[a]["test_preds"]) for a in tc.ARCHS}
display(pd.DataFrame({a: bootstrap_ci(p) for a, p in preds_rep.items()}).T)

rows = []
for i, a in enumerate(tc.ARCHS):
    for b in tc.ARCHS[i + 1:]:
        ca = preds_rep[a].y_pred == preds_rep[a].y_true
        cb = preds_rep[b].y_pred == preds_rep[b].y_true
        table = [[int((ca & cb).sum()), int((ca & ~cb).sum())], [int((~ca & cb).sum()), int((~ca & ~cb).sum())]]
        r = mcnemar(table, exact=min(table[0][1], table[1][0]) < 25)
        rows.append(dict(A=a, B=b, solo_A_acierta=table[0][1], solo_B_acierta=table[1][0], p_valor=round(float(r.pvalue), 4)))
display(pd.DataFrame(rows))

## 11. Registro de modelos y cierre

In [ ]:
# @title 11.1 Registrar el modelo representativo de cada arquitectura en el Model Registry de MLflow
for arch, info in representative.items():
    run_id = load_val(arch, info["seed"])["run_id"]
    tc.safe(mlflow.register_model, f"runs:/{run_id}/model", f"ham10000-{arch}")

with mlflow.start_run(run_name="modulo2_resumen"):
    mlflow.set_tags({"modulo": "2"})
    for f in ["resumen_test_media_std.csv", "por_clase_test_media.csv", "mel_false_negatives.csv", "representative_models.json"]:
        tc.safe(mlflow.log_artifact, str(paths.drive_results / f))
    tc.safe(mlflow.log_artifacts, str(FIG_DIR), "figuras")
print("Listo. Commit + push del notebook.")